In [1]:
import plotly.express as px
import pandas as pd
import numpy as np
from pathlib import Path
import os


In [13]:

# Localização e leitura do conjunto de dados
arquivo = Path("C:/Users/Lnk3x/Downloads/VisuDados/steam_games.csv")
limite_linhas = 100000

if not arquivo.exists():
    raise FileNotFoundError("O arquivo 'steam_games.csv' não foi encontrado.")

try:
    df = pd.read_csv(arquivo, nrows=limite_linhas)
except UnicodeDecodeError:
    df = pd.read_csv(arquivo, encoding="latin1", nrows=limite_linhas)

# Tratamento inicial de valores ausentes
valores_ausentes = ["", "NA", "N/A", "NaN", "null", "NULL", "None", "none", "tbd"]
df = df.replace(valores_ausentes, pd.NA)

print("=" * 80)
print("a) IDENTIFICAÇÃO DO CONJUNTO DE DADOS")
print("=" * 80)
print("Nome do conjunto: Steam Games")
print("Arquivo utilizado: steam_games.csv")
print("Fonte: https://www.kaggle.com/datasets/hubertsidorowicz/steam-games-dataset-daily-updates")
print("Biblioteca utilizada: pandas")
print("Categoria do conjunto de dados: multivariado e tabular, pois cada jogo é descrito por diversos atributos. Também contém dados textuais e categóricos, como nome, desenvolvedores, gêneros, categorias e tags. A data de lançamento permite análises temporais, mas o conjunto não é uma série temporal. Não é essencialmente espacial, topológico ou hierárquico.")

print("\n" + "=" * 80)
print("b) CARACTERÍSTICAS DO CONJUNTO DE DADOS")
print("=" * 80)
print(f"Amostra analisada: {df.shape[0]} linhas, com limite de {limite_linhas} linhas.")
print(f"Número de instâncias: {df.shape[0]}")
print(f"Número de atributos: {df.shape[1]}")
print("Categoria: dados tabulares, jogos digitais, análise exploratória e sistemas de recomendação.")
print(f"Memória ocupada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nTipos de dados:")
print(df.dtypes.value_counts())

print("\n" + "=" * 80)
print("c) ATRIBUTOS, TIPOS E INTERVALOS DE VALORES")
print("=" * 80)

descricoes = {
    "appid": "Identificador único do jogo.",
    "name": "Nome do jogo.",
    "release_date": "Data de lançamento.",
    "estimated_owners": "Estimativa da quantidade de proprietários.",
    "peak_ccu": "Número máximo estimado de usuários simultâneos.",
    "required_age": "Idade mínima recomendada.",
    "price": "Preço do jogo.",
    "dlc_count": "Quantidade de conteúdos adicionais.",
    "metacritic_score": "Pontuação atribuída pelo Metacritic.",
    "user_score": "Pontuação atribuída pelos usuários.",
    "positive": "Quantidade de avaliações positivas.",
    "negative": "Quantidade de avaliações negativas.",
    "achievements": "Quantidade de conquistas disponíveis.",
    "recommendations": "Número de recomendações dos usuários.",
    "average_playtime_forever": "Tempo médio total de jogo.",
    "average_playtime_two_weeks": "Tempo médio jogado nas últimas duas semanas.",
    "developers": "Desenvolvedores do jogo.",
    "publishers": "Publicadoras do jogo.",
    "categories": "Categorias e recursos disponíveis.",
    "genres": "Gêneros do jogo.",
    "tags": "Etiquetas associadas ao jogo."
}

for coluna in df.columns:
    serie = df[coluna]
    nome_normalizado = coluna.lower().strip()

    descricao = descricoes.get(
        nome_normalizado,
        "Atributo descritivo ou técnico relacionado ao jogo."
    )

    print(f"\nAtributo: {coluna}")
    print(f"Tipo: {serie.dtype}")
    print(f"Descrição: {descricao}")
    print(f"Valores distintos: {serie.nunique(dropna=True)}")

    if pd.api.types.is_numeric_dtype(serie):
        print(f"Intervalo: {serie.min()} até {serie.max()}")
    else:
        exemplos = serie.dropna().astype(str).head(3).tolist()
        print(f"Exemplos: {exemplos}")

print("\n" + "=" * 80)
print("d) VALORES AUSENTES, DUPLICATAS E PRÉ-PROCESSAMENTO")
print("=" * 80)

faltantes_por_coluna = df.isna().sum()
faltantes_por_coluna = faltantes_por_coluna[faltantes_por_coluna > 0]

print(f"Instâncias com pelo menos um valor ausente: "
      f"{df.isna().any(axis=1).sum()} "
      f"({df.isna().any(axis=1).mean() * 100:.2f}%)")

if len(faltantes_por_coluna) > 0:
    print("\nValores ausentes por atributo:")
    print(faltantes_por_coluna.sort_values(ascending=False))
else:
    print("Não foram encontrados valores ausentes.")

print(f"\nQuantidade total de valores ausentes: {df.isna().sum().sum()}")
print(f"Linhas duplicadas: {df.duplicated().sum()}")

print("\nPré-processamento sugerido:")
print("- Valores como '', NA, N/A, null e None foram convertidos para pd.NA.")
print("- Valores numéricos podem ser preenchidos posteriormente pela mediana.")
print("- Valores categóricos podem ser preenchidos pela moda ou pela categoria 'Não informado'.")
print("- Atributos textuais, como descrição e tags, podem exigir tratamento específico.")
print("- Não há uma classe-alvo evidente; portanto, o conjunto não é originalmente classificado.")
print("- Os atributos 'genres', 'categories' e 'tags' podem ser usados como rótulos "
      "em tarefas de classificação multilabel.")

# Distribuição dos gêneros, caso exista a coluna
coluna_genero = next(
    (c for c in df.columns if c.lower().strip() == "genres"),
    None
)

if coluna_genero:
    generos = (
        df[coluna_genero]
        .dropna()
        .astype(str)
        .str.split(r"[;,|]")
        .explode()
        .str.strip()
    )
    generos = generos[generos != ""]
    distribuicao = generos.value_counts()
    print("\nDistribuição dos gêneros:")
    print(distribuicao.head(20))
    print("\nPercentual dos gêneros:")
    print((distribuicao.head(20) / distribuicao.sum() * 100).round(2))

print("\n" + "=" * 80)
print("e) PERGUNTAS DE INVESTIGAÇÃO")
print("=" * 80)

perguntas = [
    "1. Existe relação entre o preço de um jogo e sua quantidade de avaliações?",
    "2. Quais gêneros possuem maior número de jogos e melhores avaliações?",
    "3. Como a quantidade de jogadores simultâneos varia ao longo dos anos?",
    "4. Jogos gratuitos apresentam maior número de avaliações ou recomendações?",
    "5. Existe relação entre o tempo médio de jogo e a avaliação dos usuários?",
    "6. Quais desenvolvedores ou publicadoras possuem maior quantidade de jogos populares?"
]

for pergunta in perguntas:
    print(pergunta)

C:\Users\Lnk3x\AppData\Local\Temp\ipykernel_25592\1242415959.py:9: DtypeWarning: Columns (0: full_audio_languages) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arquivo, nrows=limite_linhas)


a) IDENTIFICAÇÃO DO CONJUNTO DE DADOS
Nome do conjunto: Steam Games
Arquivo utilizado: steam_games.csv
Fonte: https://www.kaggle.com/datasets/hubertsidorowicz/steam-games-dataset-daily-updates
Biblioteca utilizada: pandas
Categoria do conjunto de dados: multivariado e tabular, pois cada jogo é descrito por diversos atributos. Também contém dados textuais e categóricos, como nome, desenvolvedores, gêneros, categorias e tags. A data de lançamento permite análises temporais, mas o conjunto não é uma série temporal. Não é essencialmente espacial, topológico ou hierárquico.

b) CARACTERÍSTICAS DO CONJUNTO DE DADOS
Amostra analisada: 100000 linhas, com limite de 100000 linhas.
Número de instâncias: 100000
Número de atributos: 42
Categoria: dados tabulares, jogos digitais, análise exploratória e sistemas de recomendação.
Memória ocupada: 881.33 MB

Tipos de dados:
str        23
int64      10
bool        5
float64     4
Name: count, dtype: int64

c) ATRIBUTOS, TIPOS E INTERVALOS DE VALORES

At

In [16]:
import plotly.express as px

# Criar métricas a partir das colunas originais do conjunto
for coluna in ["price", "positive", "negative", "peak_ccu"]:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

df["total_reviews"] = df["positive"].fillna(0) + df["negative"].fillna(0)
df["approval_rate"] = df["positive"].div(df["total_reviews"]).where(df["total_reviews"] > 0)

# Escalas logarítmicas exigem preço e total de avaliações maiores que zero
df_v1 = df[(df["total_reviews"] > 0) & (df["price"] > 0)].copy()
df_v1["peak_ccu"] = df_v1["peak_ccu"].fillna(0)

fig1 = px.scatter(
    df_v1,
    x="price",
    y="total_reviews",
    color="approval_rate",
    size="peak_ccu",
    hover_name="name",
    hover_data={"price": ":$.2f", "total_reviews": ":,", "approval_rate": ":.1%", "peak_ccu": ":,"},
    log_x=True,
    log_y=True,
    trendline="ols",
    trendline_color_override="red",
    title="<b>Relação entre Preço e Total de Avaliações (Escala Logarítmica)</b>",
    labels={
        "price": "Preço (USD - Escala Log)",
        "total_reviews": "Total de Avaliações (Escala Log)",
        "approval_rate": "Aprovação (%)",
        "peak_ccu": "Players Concorrentes"
    },
    color_continuous_scale="RdYlGn"
)

fig1.update_layout(template="plotly_white")
fig1.show()

# Para salvar em HTML interativo para a apresentação:
# fig1.write_html("visualizacao1_scatter_regressao.html")

In [18]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# 1. Carregar os dados
arquivo = "steam_games.csv"  # Certifique-se de que o arquivo está no mesmo diretório
df = pd.read_csv(arquivo, nrows=100000)

# 2. Tratamento inicial e remoção de nulos
df.replace(["", "NA", "N/A", "NaN", "null", "None", "none", "tbd"], np.nan, inplace=True)

df['positive'] = pd.to_numeric(df['positive'], errors='coerce').fillna(0)
df['negative'] = pd.to_numeric(df['negative'], errors='coerce').fillna(0)
df['recommendations'] = pd.to_numeric(df['recommendations'], errors='coerce').fillna(0)
df['total_reviews'] = df['positive'] + df['negative']

df['approval_rate'] = np.where(
    df['total_reviews'] > 0, 
    (df['positive'] / df['total_reviews']) * 100, 
    0
)

# 3. Tratamento da coluna de desenvolvedores
# Filtra desenvolvedores ausentes ou vazios
df_devs = df.dropna(subset=['developers']).copy()
df_devs['developers'] = (
    df_devs['developers']
    .astype(str)
    .str.replace(r'[\[\]"]', '', regex=True) # Limpa formato de lista JSON/string
    .str.strip()
)

# 4. Agregação dos dados por Desenvolvedor
# Filtramos apenas jogos populares (ex: com mais de 1.000 avaliações ou alta contagem de jogos)
dev_stats = df_devs.groupby('developers').agg(
    total_jogos=('name', 'count'),
    jogos_populares=('total_reviews', lambda x: (x >= 1000).sum()),
    media_recomendacoes=('recommendations', 'mean'),
    taxa_aprovacao_media=('approval_rate', 'mean')
).reset_index()

# Filtrar Top 20 Desenvolvedores com mais jogos populares (ou mais jogos cadastrados)
top20_devs = (
    dev_stats
    .sort_values(by='jogos_populares', ascending=True)  # Ascendente para plotar de cima para baixo no eixo Y
    .tail(20)
)

# ==============================================================================
# VISUALIZAÇÃO 2 (PERGUNTA 6): Lollipop Chart para Top 20 Desenvolvedores
# ==============================================================================
fig = go.Figure()

# 1. Haste do Pirulito (Linhas horizontais do zero até o valor)
for i, row in top20_devs.iterrows():
    fig.add_trace(go.Scatter(
        x=[0, row['jogos_populares']],
        y=[row['developers'], row['developers']],
        mode='lines',
        line=dict(color='gray', width=2),
        showlegend=False,
        hoverinfo='none'
    ))

# 2. Cabeça do Pirulito (Círculo na ponta)
# O tamanho representa a média de recomendações e a cor representa a taxa de aprovação
fig.add_trace(go.Scatter(
    x=top20_devs['jogos_populares'],
    y=top20_devs['developers'],
    mode='markers',
    name='Desenvolvedores',
    marker=dict(
        size=top20_devs['media_recomendacoes'] / top20_devs['media_recomendacoes'].max() * 25 + 10, # Escala visual de tamanho
        color=top20_devs['taxa_aprovacao_media'],
        colorscale='RdYlGn',
        cmin=50, cmax=100,
        colorbar=dict(title="Taxa Média de<br>Aprovação (%)"),
        showscale=True,
        line=dict(color='black', width=1)
    ),
    text=top20_devs['developers'],
    customdata=np.stack((
        top20_devs['total_jogos'], 
        top20_devs['jogos_populares'], 
        top20_devs['media_recomendacoes'],
        top20_devs['taxa_aprovacao_media']
    ), axis=-1),
    hovertemplate=(
        "<b>%{text}</b><br><br>" +
        "Jogos Populares (>=1k reviews): %{customdata[1]}<br>" +
        "Total de Jogos no Dataset: %{customdata[0]}<br>" +
        "Média de Recomendações: %{customdata[2]:,.0f}<br>" +
        "Taxa de Aprovação Média: %{customdata[3]:.1f}%" +
        "<extra></extra>"
    )
))

# Layout e Estilização do Gráfico
fig.update_layout(
    title="<b>Top 20 Desenvolvedores com Maior Quantidade de Jogos Populares na Steam</b>",
    xaxis_title="Quantidade de Jogos Populares (≥ 1.000 avaliações)",
    yaxis_title="Desenvolvedor",
    template="plotly_white",
    height=700,
    margin=dict(l=150, r=50, t=80, b=50)
)

fig.show()

# Para salvar o gráfico interativo para a apresentação:
# fig.write_html("visualizacao2_lollipop_desenvolvedores.html")

C:\Users\Lnk3x\AppData\Local\Temp\ipykernel_25592\3687378370.py:7: DtypeWarning: Columns (0: full_audio_languages) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arquivo, nrows=100000)


In [35]:
import pandas as pd
import numpy as np
import plotly.express as px

# Carregar e preparar os dados
arquivo = "steam_games.csv"
df = pd.read_csv(arquivo, nrows=500000)

valores_ausentes = ["", "NA", "N/A", "NaN", "null", "None", "none", "tbd"]
df.replace(valores_ausentes, np.nan, inplace=True)

colunas_num = ["price", "positive", "negative", "peak_ccu", "recommendations"]
for coluna in colunas_num:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

df["total_reviews"] = df["positive"].fillna(0) + df["negative"].fillna(0)
df["approval_rate"] = (
    df["positive"].div(df["total_reviews"]).mul(100)
    .where(df["total_reviews"] > 0)
)

# Comparar as maiores e menores aprovações entre os 50 jogos com maior peak_ccu
df_candidatos = df[
    df["peak_ccu"].gt(0)
    & df["total_reviews"].gt(0)
    & df["name"].notna()
].nlargest(1000, "peak_ccu")

jogos_maior_aprovacao = df_candidatos.nlargest(100, "approval_rate").copy()
jogos_maior_aprovacao["grupo"] = "Maior aprovação"

jogos_menor_aprovacao = df_candidatos.nsmallest(100, "approval_rate").copy()
jogos_menor_aprovacao["grupo"] = "Menor aprovação"

jogos_selecionados = pd.concat(
    [jogos_maior_aprovacao, jogos_menor_aprovacao]
).drop_duplicates(subset="name")

fig = px.scatter(
    jogos_selecionados,
    x="approval_rate",
    y="peak_ccu",
    size="total_reviews",
    size_max=75,
    color="approval_rate",
    color_continuous_scale="RdYlGn",
    range_color=[0, 100],
    hover_name="name",
    hover_data={
        "approval_rate": ":.1f",
        "peak_ccu": ":,",
        "total_reviews": ":,",
        "price": ":.2f",
        "grupo": True
    },
    title="<b>Jogos com maior peak_ccu: maiores e menores taxas de aprovação</b>",
    labels={
        "approval_rate": "Taxa de aprovação (%)",
        "peak_ccu": "Pico de jogadores simultâneos",
        "total_reviews": "Total de avaliações",
        "grupo": "Grupo"
    }
)

fig.update_traces(
    marker=dict(opacity=0.82, line=dict(width=1, color="#333333"))
)
fig.update_layout(
    template="plotly_white",
    height=650,
    legend_title_text="Taxa de aprovação"
)

fig.show()

# Para salvar o gráfico interativo para a apresentação:
# fig.write_html("visualizacao_bolhas_peak_ccu_aprovacao.html")

C:\Users\Lnk3x\AppData\Local\Temp\ipykernel_25592\3032602097.py:7: DtypeWarning: Columns (0: windows, 1: mac, 2: linux) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arquivo, nrows=500000)


In [24]:


# 1. Carregar os dados
arquivo = "steam_games.csv"  # Certifique-se de que o arquivo está no mesmo diretório
df = pd.read_csv(arquivo, nrows=500000)

# 2. Tratamento inicial e conversão numérica
valores_ausentes = ["", "NA", "N/A", "NaN", "null", "None", "none", "tbd"]
df.replace(valores_ausentes, np.nan, inplace=True)

colunas_num = ['price', 'positive', 'negative', 'peak_ccu', 'recommendations']
for col in colunas_num:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Métricas derivadas
df['total_reviews'] = df['positive'] + df['negative']
df['approval_rate'] = np.where(
    df['total_reviews'] > 0, 
    (df['positive'] / df['total_reviews']) * 100, 
    0
)

# 3. Tratamento da coluna de gêneros (explodir jogos com múltiplos gêneros)
df_genres = df.dropna(subset=['genres']).copy()

df_genres['genres_clean'] = (
    df_genres['genres']
    .astype(str)
    .str.replace(r'[\[\]"]', '', regex=True)
    .str.split(',')
)

df_exploded = df_genres.explode('genres_clean')
df_exploded['genres_clean'] = df_exploded['genres_clean'].str.strip()

# Remover resíduos de formatação JSON e categorias vazias
df_exploded = df_exploded[~df_exploded['genres_clean'].str.startswith('{')]
df_exploded = df_exploded[df_exploded['genres_clean'] != '']

# 4. Agregação dos dados por Gênero
genre_stats = df_exploded.groupby('genres_clean').agg(
    total_jogos=('name', 'count'),
    taxa_aprovacao_media=('approval_rate', 'mean'),
    media_preco=('price', 'mean'),
    total_avaliacoes=('total_reviews', 'sum')
).reset_index()

# Filtrar gêneros significativos (com mais de 50 jogos cadastrados)
genre_stats = genre_stats[genre_stats['total_jogos'] >= 50]

# ==============================================================================
# VISUALIZAÇÃO 2 (PERGUNTA 2): Treemap de Gêneros
# ==============================================================================
fig = px.treemap(
    genre_stats,
    path=[px.Constant("Todos os Gêneros"), 'genres_clean'],
    values='total_jogos',
    color='taxa_aprovacao_media',
    color_continuous_scale='RdYlGn',
    range_color=[30, 60],  # Ajuste da escala de cor para evidenciar diferenças
    hover_name='genres_clean',
    title="<b>Distribuição de Gêneros na Steam: Volume de Jogos vs. Taxa de Aprovação Média</b>",
    labels={
        "total_jogos": "Quantidade de Jogos",
        "taxa_aprovacao_media": "Taxa Média de Aprovação (%)",
        "genres_clean": "Gênero",
        "media_preco": "Preço Médio (USD)",
        "total_avaliacoes": "Total de Avaliações"
    },
    custom_data=['media_preco', 'total_avaliacoes']
)

# Ajuste do conteúdo do Tooltip (Hover)
fig.update_traces(
    hovertemplate=(
        "<b>Gênero: %{label}</b><br><br>" +
        "Quantidade Total de Jogos: %{value:,.0f}<br>" +
        "Taxa Média de Aprovação: %{color:.1f}%<br>" +
        "Preço Médio: $%{customdata[0]:.2f}<br>" +
        "Total de Avaliações Acumuladas: %{customdata[1]:,.0f}" +
        "<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    height=650,
    margin=dict(t=50, l=10, r=10, b=10),
    coloraxis_colorbar=dict(title="Aprovação<br>Média (%)")
)

fig.show()

# Para salvar o gráfico em HTML interativo para a apresentação:
# fig.write_html("visualizacao2_treemap_generos.html")

C:\Users\Lnk3x\AppData\Local\Temp\ipykernel_25592\3653886951.py:3: DtypeWarning: Columns (0: windows, 1: mac, 2: linux) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arquivo, nrows=500000)
